In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import os

# --- PART 1: EDA & FEATURE ENGINEERING ---

# 1. Load the dataset
# Note: Path adjusted to assume you are running this from the project root
print("Loading dataset...")
data_path = 'data/raw/creditcard.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}. Please ensure it is in place.")

df = pd.read_csv(data_path)

# 2. Basic EDA
print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution (0 = Normal, 1 = Fraud):")
print(df['Class'].value_counts(normalize=True) * 100)

# 3. Feature Engineering
print("\nEngineering new features...")

# Feature 1: Time_Hour (Convert elapsed seconds to hour of the day)
df['Time_Hour'] = (df['Time'] / 3600) % 24

# Feature 2: Amount_Log (Logarithmic transformation of Amount)
df['Amount_Log'] = np.log1p(df['Amount'])

# Feature 3: Is_High_Amount (Binary flag for transactions > $200)
df['Is_High_Amount'] = (df['Amount'] > 200).astype(int)

# 4. Drop original raw columns
df_processed = df.drop(columns=['Time', 'Amount'])

print(f"Processed Dataset Shape: {df_processed.shape}")
print("Feature Engineering Complete.")


# --- PART 2: DATA SPLITTING & EXPORT ---

# 1. Separate features (X) and target (y)
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

# 2. Split into training and testing sets (80% train, 20% test)
# Stratification ensures the 0.17% fraud ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Fraud cases in training set: {sum(y_train == 1)}")

# 3. Save the PRE-SMOTE datasets
# We do NOT apply SMOTE here to prevent data leakage.
# SMOTE is handled dynamically during training.

os.makedirs('data/processed', exist_ok=True)

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('data/processed/train.csv', index=False)
test_data.to_csv('data/processed/test_data.csv', index=False)

print("\nData processing complete. Files saved to data/processed/")
print("Ready for dynamic SMOTE and model training in train.py!")

Loading dataset...
Dataset Shape: (284807, 31)

Class Distribution (0 = Normal, 1 = Fraud):
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

Engineering new features...
Processed Dataset Shape: (284807, 32)
Feature Engineering Complete. New features added: 'Time_Hour', 'Amount_Log', 'Is_High_Amount'


In [ ]:
from sklearn.model_selection import train_test_split
import os

# 1. Separate features (X) and target (y)
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

# 2. Split into training and testing sets (80% train, 20% test)
# Using stratify=y is crucial here to maintain the 0.17% fraud ratio in both sets!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Fraud cases in training set: {sum(y_train == 1)}")

# 3. Save the PRE-SMOTE datasets
# We do NOT apply SMOTE here. We save the raw training data so that SMOTE 
# can be applied dynamically inside the cross-validation pipeline in train.py 
# to prevent data leakage!

os.makedirs('../data/processed', exist_ok=True)

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('../data/processed/train.csv', index=False)
test_data.to_csv('../data/processed/test_data.csv', index=False)

print("Data processing complete. Files saved to data/processed/")
print("Ready for dynamic SMOTE and model training in train.py!")

Training set shape before SMOTE: (227845, 31)
Fraud cases in training set before SMOTE: 394
Training set shape after SMOTE: (454902, 31)
Fraud cases in training set after SMOTE: 227451
Data processing complete. Files saved to data/processed/
